# 🛡️ 企業 AI 工具可疑下載來源偵測器

> **目的**：搜尋並評分可疑 AI 工具下載來源，輸出企業黑名單 TOP 20 及多格式防火牆封鎖清單。
>
> **使用方式**：由上往下依序執行每個儲存格（`Shift+Enter`），或點選 **執行階段 → 全部執行**。

| 儲存格 | 功能 |
|--------|------|
| 1 | 安裝相依套件 |
| 2 | 匯入套件與設定參數 |
| 3 | 載入評分規則資料庫 |
| 4 | 定義核心評分函數 |
| 5 | 執行偵測，輸出 TOP 20 黑名單 |
| 6 | 下載防火牆封鎖清單（多格式） |

In [ ]:
# ── 儲存格 1：安裝相依套件 ─────────────────────────────────
!pip install googlesearch-python colorama tabulate -q
print("✅ 套件安裝完成")


In [ ]:
# ── 儲存格 2：匯入套件與設定參數 ───────────────────────────
import time, re, urllib.parse
from datetime import datetime
from tabulate import tabulate
from colorama import Fore, Style, init
init(autoreset=True)

try:
    from googlesearch import search as google_search
    GOOGLE_AVAILABLE = True
    print("✅ Google 搜尋模組載入成功")
except ImportError:
    GOOGLE_AVAILABLE = False
    print("⚠️  Google 搜尋模組未載入，將使用靜態威脅情資模式")

# ── 搜尋關鍵字（模擬員工可能輸入的搜尋詞）
SEARCH_QUERIES = [
    "ChatGPT 中文版 下載",
    "ChatGPT desktop 安裝 免費",
    "ChatGPT china download",
    "chatgpt unofficial download windows",
    "chatgpt破解版下載",
    "gpt4 免費桌面版",
    "ChatGPT bypass 下載",
    "whensunset chatgpt download",
    "chatgpt-china install",
]

# ── 官方白名單（搜尋結果中排除這些來源）
OFFICIAL_WHITELIST = [
    "openai.com", "chat.openai.com", "anthropic.com", "claude.ai",
    "microsoft.com", "copilot.microsoft.com", "gemini.google.com",
    "google.com", "bing.com", "yahoo.com", "github.com/openai",
    "apps.apple.com", "play.google.com", "wikipedia.org",
    "support.openai.com",
]

print(f"✅ 設定完成：{len(SEARCH_QUERIES)} 組搜尋查詢，{len(OFFICIAL_WHITELIST)} 筆白名單")


In [ ]:
# ── 儲存格 3：定義評分規則資料庫 ───────────────────────────

# 可疑關鍵字黑名單評分
SUSPICIOUS_KEYWORDS = {
    # 中文可疑詞
    "中文版": 10, "破解": 25, "免費版": 10, "無限制": 20,
    "最新版":  8, "下載安裝": 12, "中國版": 30, "越牆": 35,
    "無需翻牆": 30, "不限流量": 20, "免費下載": 15,
    # 英文可疑詞
    "china": 20, "unofficial": 25, "bypass": 30, "free-download": 20,
    "crack": 40, "pro-free": 35, "unlimited": 15, "no-vpn": 30,
    "desktop-free": 20, "chatgpt-go": 40, "openai-free": 35,
    "gpt-china": 45, "whensunset": 50, "chatgpt-china": 50,
    # 檔名模式
    "setup.exe": 30, "installer.exe": 25, ".gz": 20,
}

# 可疑域名正規表示式模式
SUSPICIOUS_DOMAIN_PATTERNS = [
    (r"chat.?gpt(?!\.openai)",         "冒用 chatgpt 域名"),
    (r"openai(?!\.com)",               "冒用 openai 品牌"),
    (r"gpt.?(china|cn|free|pro)",       "GPT + 中國/免費關鍵字"),
    (r"(free|crack|bypass).?gpt",       "破解/免費 GPT"),
    (r"gpt.?(bypass|crack|unlimited)",  "GPT 繞過/破解"),
    (r"ai.?download",                   "AI 下載站"),
    (r"chatbot.*(free|download)",       "免費聊天機器人下載"),
]

# 已知惡意域名資料庫（來自 Cyble / Malwarebytes / TrendMicro 威脅情資）
KNOWN_MALICIOUS_DOMAINS = {
    "chatgpt-go.online":         ("Lumma Stealer 散布站",        100),
    "chat-gpt-online-pc.com":    ("Aurora Stealer 下載點",       100),
    "chatgptfreeapp.com":        ("已知釣魚頁面",                  90),
    "chat.chatbotapp.ai":        ("非官方仿冒介面，資料收集站",     70),
    "openai-chatgpt.online":     ("仿冒 OpenAI 釣魚站",           85),
    "chatgpt4free.io":           ("免費誘餌釣魚站",               80),
    "gpt4-download.com":         ("惡意安裝檔散布站",             95),
    "chatgpt-pro.download":      ("惡意 Pro 版誘餌",              90),
    "chatgpt-china.net":         ("中國版偽裝惡意軟體",           100),
    "whensunset.chatgpt-cn.com": ("本次事件已確認惡意來源",       100),
    "ai-chatgpt-free.com":       ("仿冒 AI 工具下載站",           85),
    "gpt-unlimited.online":      ("無限制版誘餌站",               80),
    "chatgpt-bypass.net":        ("VPN 繞過工具偽裝",             95),
    "free-chatgpt-download.com": ("桌面版釣魚安裝程式",           88),
    "chatgpt-desktop-cn.com":    ("台灣定向攻擊站",               92),
    "chatgpt-online.net":        ("仿冒 ChatGPT 線上版",          75),
    "gpt4online.com":            ("未授權 GPT-4 存取",            70),
    "chatgptx.download":         ("惡意下載站",                   90),
    "openai-gpt.download":       ("仿冒 OpenAI 下載頁",           88),
    "aichatbot-free.com":        ("免費 AI 誘餌站",               72),
}

# 高風險 TLD 評分表
RISKY_TLDS = {
    ".online": 15, ".xyz": 15, ".cn": 20, ".tk": 25,
    ".top": 10, ".club": 10, ".download": 30, ".icu": 20,
    ".site": 12, ".live": 12,
}

print(f"✅ 資料庫載入完成")
print(f"   可疑關鍵字  : {len(SUSPICIOUS_KEYWORDS)} 筆")
print(f"   域名模式    : {len(SUSPICIOUS_DOMAIN_PATTERNS)} 組")
print(f"   已知惡意域名: {len(KNOWN_MALICIOUS_DOMAINS)} 筆")
print(f"   高風險 TLD  : {len(RISKY_TLDS)} 種")


In [ ]:
# ── 儲存格 4：定義核心評分函數 ─────────────────────────────

def score_url(url: str, title: str = "", snippet: str = ""):
    """對 URL 進行多維度可疑度評分，回傳 None 表示白名單排除"""
    score = 0
    flags = []

    try:
        parsed = urllib.parse.urlparse(url)
        domain = parsed.netloc.lower().replace("www.", "")
        path   = parsed.path.lower()
        full   = (url + " " + title + " " + snippet).lower()
    except Exception:
        return None

    # 1. 官方白名單 → 排除
    for white in OFFICIAL_WHITELIST:
        if white in domain:
            return None

    # 2. 已知惡意域名（直接最高風險）
    for mal_domain, (reason, mal_score) in KNOWN_MALICIOUS_DOMAINS.items():
        if mal_domain in domain:
            score += mal_score
            flags.append(f"⛔ 已知惡意: {reason}")

    # 3. 可疑關鍵字評分
    for kw, pts in SUSPICIOUS_KEYWORDS.items():
        if kw in full:
            score += pts
            flags.append(f"🔑 可疑詞: '{kw}' (+{pts})")

    # 4. 域名模式比對
    for pattern, desc in SUSPICIOUS_DOMAIN_PATTERNS:
        if re.search(pattern, domain):
            score += 20
            flags.append(f"🌐 域名警示: {desc}")

    # 5. 高風險 TLD
    for tld, pts in RISKY_TLDS.items():
        if domain.endswith(tld):
            score += pts
            flags.append(f"🔴 高風險TLD: {tld} (+{pts})")

    # 6. 路徑中含可執行檔副檔名
    for ext in [".exe", ".msi", ".dmg", ".pkg", ".gz", ".zip", ".bat"]:
        if ext in path:
            score += 25
            flags.append(f"📦 路徑含可執行檔: {ext}")

    # 7. 非 HTTPS
    if not url.startswith("https://"):
        score += 15
        flags.append("🔓 非 HTTPS 連線 (+15)")

    # 8. 數字堆疊域名
    if re.search(r"\d{6,}", domain):
        score += 20
        flags.append("🔢 域名含大量數字（垃圾站特徵）")

    # 風險等級
    if   score >= 80: risk = "🚨 極高風險"
    elif score >= 50: risk = "⚠️  高風險"
    elif score >= 25: risk = "🔶 中風險"
    else:             risk = "ℹ️  低風險"

    return {"url": url, "domain": domain, "title": (title[:60] if title else domain),
            "score": score, "flags": flags, "risk": risk}


def collect_urls(queries, results_per_query=6):
    """執行搜尋並收集 URL"""
    collected = []
    if GOOGLE_AVAILABLE:
        print(f"🔍 開始搜尋引擎爬取（{len(queries)} 組查詢）\n")
        for query in queries:
            print(f"  🔎 搜尋: {query}")
            try:
                urls = list(google_search(query, num_results=results_per_query, lang="zh-TW"))
                print(f"       → 取得 {len(urls)} 筆")
                for url in urls:
                    collected.append({"url": url, "query": query, "title": "", "snippet": ""})
                time.sleep(2)
            except Exception as e:
                print(f"       ⚠ 搜尋失敗: {e}")
    else:
        print("⚠️  靜態模式：跳過即時搜尋，使用威脅情資資料庫")

    print(f"\n📋 載入威脅情資資料庫（{len(KNOWN_MALICIOUS_DOMAINS)} 筆）")
    for domain in KNOWN_MALICIOUS_DOMAINS:
        collected.append({"url": f"https://{domain}/download",
                          "query": "威脅情資資料庫", "title": domain, "snippet": ""})
    return collected


def generate_blocklist(results):
    """產生多格式防火牆封鎖清單"""
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    lines = [
        "# ============================================================",
        f"# 企業 AI 工具可疑下載域名封鎖清單",
        f"# 產生時間: {ts}",
        f"# 共 {len(results)} 筆（風險分數 ≥ 50）",
        "# ============================================================", "",
        "# --- Squid Proxy 格式 ---",
    ]
    for r in results:
        lines.append(f"acl blacklist_ai dstdomain .{r['domain']}  # score={r['score']}")
    lines += ["", "# --- iptables 格式 ---"]
    for r in results:
        lines.append(f"iptables -I FORWARD -d {r['domain']} -j DROP  # score={r['score']}")
    lines += ["", "# --- Windows hosts 檔格式 ---"]
    for r in results:
        lines.append(f"0.0.0.0  {r['domain']}  # score={r['score']}")
    lines += ["", "# --- Fortinet 格式 ---"]
    for r in results:
        lines += [f"config firewall address",
                  f"    edit \"{r['domain']}\"",
                  f"        set type fqdn",
                  f"        set fqdn {r['domain']}",
                  f"    next", "end"]
    return "\n".join(lines)

print("✅ 函數定義完成")


In [ ]:
# ── 儲存格 5：執行偵測與輸出黑名單 TOP 20 ──────────────────
print("\n" + "="*65)
print("  🛡️  企業 AI 工具可疑下載來源偵測器 v1.0")
print(f"  執行時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*65 + "\n")

# Step 1: 收集 URL
raw = collect_urls(SEARCH_QUERIES, results_per_query=6)

# Step 2: 評分與去重
print(f"\n⚙️  分析 {len(raw)} 筆 URL 中...")
scored, seen = [], set()
for item in raw:
    r = score_url(item["url"], item.get("title",""), item.get("snippet",""))
    if r is None or r["score"] < 10 or r["domain"] in seen:
        continue
    seen.add(r["domain"])
    r["source_query"] = item.get("query","")
    scored.append(r)

# Step 3: 排序取前 20
top20 = sorted(scored, key=lambda x: x["score"], reverse=True)[:20]

# Step 4: 輸出 TOP 20 表格
print("\n" + "="*80)
print("  🚫 企業 AI 工具可疑下載黑名單 TOP 20")
print("  （已排除官方來源：OpenAI / Anthropic / Microsoft / Google）")
print("="*80 + "\n")

table_data = []
for i, r in enumerate(top20, 1):
    main_flag = r["flags"][0] if r["flags"] else "—"
    table_data.append([f"{i:02d}", r["domain"][:42], r["score"],
                       r["risk"], main_flag[:48]])

print(tabulate(table_data,
    headers=["#", "Domain", "分數", "風險等級", "主要警示"],
    tablefmt="rounded_outline",
    colalign=("center","left","center","left","left")))

# Step 5: 極高風險詳細分析
print("\n" + "─"*80)
print("  🔬 極高風險項目詳細分析（分數 ≥ 80）")
print("─"*80)
critical = [r for r in top20 if r["score"] >= 80]
if critical:
    for r in critical:
        print(f"\n  🌐 {r['domain']}")
        print(f"     URL   : {r['url'][:72]}")
        print(f"     分數  : {r['score']}  |  等級: {r['risk']}")
        print(f"     來源  : {r['source_query']}")
        print(f"     警示  :")
        for flag in r["flags"][:5]:
            print(f"            {flag}")
else:
    print("  本次掃描未發現極高風險站點")


In [ ]:
# ── 儲存格 6：輸出防火牆封鎖清單並下載 ────────────────────
high_risk = [r for r in top20 if r["score"] >= 50]
blocklist_txt = generate_blocklist(high_risk)

BLOCKLIST_FILE = "ai_suspicious_blocklist.txt"
with open(BLOCKLIST_FILE, "w", encoding="utf-8") as f:
    f.write(blocklist_txt)

# 統計摘要
print("\n" + "="*65)
print("  📊 掃描摘要")
print("="*65)
print(f"  總掃描 URL 數 : {len(raw)}")
print(f"  去重後分析數  : {len(scored)}")
print(f"  🚨 極高風險   : {sum(1 for r in top20 if r['score'] >= 80)} 筆（建議立即封鎖）")
print(f"  ⚠️  高風險     : {sum(1 for r in top20 if 50 <= r['score'] < 80)} 筆（建議加入監控）")
print(f"  🔶 中風險     : {sum(1 for r in top20 if 25 <= r['score'] < 50)} 筆（建議持續觀察）")
print(f"\n  📋 建議行動：")
print(f"     → 分數 ≥ 80：立即加入 Proxy / 防火牆黑名單")
print(f"     → 分數 ≥ 50：加入 SIEM 告警規則")
print(f"     → 分數 ≥ 25：加入員工安全教育案例")
print(f"\n  報告完成時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*65)

# 觸發 Colab 下載
from google.colab import files
files.download(BLOCKLIST_FILE)
print(f"\n  ⬇️  封鎖清單已下載：{BLOCKLIST_FILE}")
